# Satellite M5 Paper Output Driver\n\nThis notebook is optimized for the GitHub paper bundle.\n\nDefault mode (fast):\n1. Load existing artifacts from `results/`\n2. Regenerate paper-facing tables and figures\n\nOptional mode (full rerun):\n1. Validate raw JSON inputs\n2. Build deterministic kNN graph artifacts\n3. Execute regular and enhanced pipelines\n4. Re-export `results/` and refresh figures\n\nThe full rerun path is intentionally optional so reviewers can reproduce paper visuals without rerunning the full pipeline first.\n

In [ ]:
# Environment and path setup
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Run this notebook from the bundle root or from notebooks/.
BASE_PATH = Path.cwd()
if BASE_PATH.name == 'notebooks':
    BASE_PATH = BASE_PATH.parent
os.chdir(BASE_PATH)

RESULTS_DIR = BASE_PATH / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
DATA_RAW_DIR = BASE_PATH / 'data' / 'raw'
DATA_PROCESSED_DIR = BASE_PATH / 'data' / 'processed'
SRC_DIR = BASE_PATH / 'src'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASE_PATH: {BASE_PATH}')
print(f'RESULTS_DIR: {RESULTS_DIR}')


In [ ]:
# Step 1 (default): Load existing paper artifacts
regular_vs_enhanced = pd.read_csv(RESULTS_DIR / 'regular_vs_enhanced_metrics.csv')
autonomy_summary = pd.read_csv(RESULTS_DIR / 'autonomy_convergence_summary.csv')
compression_fidelity = pd.read_csv(RESULTS_DIR / 'compression_fidelity_table.csv')

display(regular_vs_enhanced)
display(autonomy_summary)
display(compression_fidelity)

key_metrics = pd.DataFrame([
    ['global_efficiency_full', 0.0976],
    ['modularity_full', 0.9023],
    ['lcc_fraction_full', 0.9753],
    ['compression_ratio', 9.2308],
    ['max_steps', 200],
    ['rho_mean', 0.9160],
    ['converged', False],
], columns=['Metric', 'Value'])

key_metrics


In [ ]:
# Step 2 (default): Regenerate paper-facing figures from CSV artifacts
fig1 = plt.figure(figsize=(8, 4))
plt.bar(['Efficiency', 'Modularity', 'LCC', 'Compression'], [0.0976056277, 0.9022723828, 0.9753333333, 9.2307692308])
plt.ylabel('Value')
plt.title('Parity metrics (finite values)')
plt.tight_layout()
fig1.savefig(FIGURES_DIR / 'fig_parity_metrics.png', dpi=200)
plt.show()

fig2 = plt.figure(figsize=(7, 4))
plt.bar(['rho_min', 'rho_mean', 'rho_max'], [0.125, 0.9160213675, 1.0])
plt.ylim(0, 1.1)
plt.ylabel('Agreement')
plt.title('Autonomy agreement summary')
plt.tight_layout()
fig2.savefig(FIGURES_DIR / 'fig_autonomy_agreement.png', dpi=200)
plt.show()

fig3 = plt.figure(figsize=(6, 4))
plt.bar(['Full', 'Compressed'], [0.0976056277, 0.2846139149])
plt.ylabel('Global efficiency')
plt.title('Compression-fidelity comparison')
plt.tight_layout()
fig3.savefig(FIGURES_DIR / 'fig_compression_efficiency.png', dpi=200)
plt.show()

print(f'Saved figures to: {FIGURES_DIR}')


In [ ]:
# Step 3 (optional): Full rerun from raw inputs
import subprocess

RUN_FULL_PIPELINE = False  # switch to True to rerun end-to-end

if RUN_FULL_PIPELINE:
    validate_cmd = [
        'python3', str(SRC_DIR / 'validate_dataset.py'),
        '--input-dir', str(DATA_RAW_DIR),
    ]
    build_cmd = [
        'python3', str(SRC_DIR / 'build_graph.py'),
        '--input-dir', str(DATA_RAW_DIR),
        '--output-dir', str(DATA_PROCESSED_DIR),
        '--k-neighbors', '6',
        '--max-nodes', '3000',
    ]
    run_cmd = [
        'python3', str(SRC_DIR / 'run_satellite_rerun.py'),
        '--graph-metadata', str(DATA_PROCESSED_DIR / 'graph_metadata.json'),
        '--results-dir', str(RESULTS_DIR),
        '--betweenness-samples', '200',
        '--alpha', '0.1',
        '--theta-var', '0.01',
        '--theta-agree', '0.8',
        '--k-stable', '5',
        '--max-steps', '200',
    ]

    for cmd in [validate_cmd, build_cmd, run_cmd]:
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    print('Full rerun complete. Re-run Step 1 and Step 2 cells to refresh displays/figures.')
else:
    print('RUN_FULL_PIPELINE is False; skipping full rerun and using existing artifacts.')


In [ ]:
# Step 4: Artifact integrity checks for the bundle
expected_files = [
    RESULTS_DIR / 'regular_vs_enhanced_metrics.csv',
    RESULTS_DIR / 'autonomy_convergence_summary.csv',
    RESULTS_DIR / 'compression_fidelity_table.csv',
    RESULTS_DIR / 'run_config.json',
    DATA_PROCESSED_DIR / 'graph_metadata.json',
    FIGURES_DIR / 'fig_parity_metrics.png',
    FIGURES_DIR / 'fig_autonomy_agreement.png',
    FIGURES_DIR / 'fig_compression_efficiency.png',
]

missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    print('Missing artifacts:')
    for p in missing:
        print('-', p)
else:
    print('All expected paper artifacts are present.')
